In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_19/data'
print(os.listdir(data_dir))

['players.csv', 'events.csv', 'teams.csv', 'matches.csv']


In [2]:
import pandas as pd
events_df = pd.read_csv(os.path.join(data_dir, 'events.csv'))
matches_df = pd.read_csv(os.path.join(data_dir, 'matches.csv'))
print(events_df.head())
print(matches_df.head())

  event_id match_id team_id player_id  minute   event_type
0    E0001     M001      TA     TA_P1       9         goal
1    E0002     M001      TA     TA_P5      35         goal
2    E0003     M001      TB     TB_P5      20         goal
3    E0004     M001      TB     TB_P2      13  yellow_card
4    E0005     M001      TA     TA_P3      22  yellow_card
  match_id  match_date home_team away_team     venue  attendance
0     M001  2024-03-04        TA        TB  Stadium2        5317
1     M002  2024-03-07        TC        TD  Stadium3        5634
2     M003  2024-03-10        TA        TC  Stadium1        5951
3     M004  2024-03-13        TB        TD  Stadium2        6268
4     M005  2024-03-16        TA        TD  Stadium3        6585


In [3]:
first_half_goals = events_df[(events_df['event_type'] == 'goal') & (events_df['minute'] <= 45)]
goals_by_match_team = first_half_goals.groupby(['match_id', 'team_id']).size().reset_index(name='goals')

# Initialize a dictionary to keep track of team stats
team_stats = {}
for team in pd.concat([matches_df['home_team'], matches_df['away_team']]).unique():
    team_stats[team] = {'points': 0, 'gd': 0, 'gf': 0}

for _, match in matches_df.iterrows():
    match_id = match['match_id']
    home_team = match['home_team']
    away_team = match['away_team']
    
    home_goals = goals_by_match_team[(goals_by_match_team['match_id'] == match_id) & (goals_by_match_team['team_id'] == home_team)]['goals'].sum()
    away_goals = goals_by_match_team[(goals_by_match_team['match_id'] == match_id) & (goals_by_match_team['team_id'] == away_team)]['goals'].sum()
    
    team_stats[home_team]['gf'] += home_goals
    team_stats[away_team]['gf'] += away_goals
    
    team_stats[home_team]['gd'] += (home_goals - away_goals)
    team_stats[away_team]['gd'] += (away_goals - home_goals)
    
    if home_goals > away_goals:
        team_stats[home_team]['points'] += 3
    elif home_goals < away_goals:
        team_stats[away_team]['points'] += 3
    else:
        team_stats[home_team]['points'] += 1
        team_stats[away_team]['points'] += 1

stats_df = pd.DataFrame.from_dict(team_stats, orient='index').reset_index()
stats_df.columns = ['team_id', 'points', 'gd', 'gf']
stats_df = stats_df.sort_values(by=['points', 'gd', 'gf', 'team_id'], ascending=[False, False, False, True])
print(stats_df)


  team_id  points  gd  gf
0      TA      10   1   6
1      TC       9   1   4
2      TB       7   0   6
3      TD       7  -2   6
